# Abdomen CT — Swin Transformer Multi-Label Sınıflandırma
### Kaggle Notebook (Birincil) · Google Colab (İkincil)

**Görev:** 6 acil karın patolojisini kesit bazında multi-label sınıflandırma  
**Model:** `swin_base_patch4_window12_384` (ImageNet-22k pretrained, 87M parametre)  
**Girdi:** DICOM → 3-kanallı HU pencereli NPZ (soft_tissue · liver · calcified)  

**Kaggle'da çalıştırma:**
1. Sağ panelden Dataset ekle: `ramazan2020/abdomen-acikveri`
2. `Settings → Accelerator → GPU (P100/T4)` seç
3. Hücreleri sırayla çalıştır

**Colab'da çalıştırma:**
1. `Runtime → Change runtime type → GPU`
2. Kaggle Secrets'a `KAGGLE_USERNAME` + `KAGGLE_KEY` ekle

---

| Adım | Süre (T4) |
|------|----------|
| NPZ Export (DICOM → 3ch float16) | ~40–60 dk |
| Eğitim (50 epoch, batch=16) | ~3–4 saat |
| Değerlendirme | ~5 dk |

> **Session kesilirse:** `SWIN_CONTINUE = True` yapıp NPZ adımını atlayın, eğitime devam edin.

---
## 0. Ortam Tespiti ve GPU Kontrolü

In [ ]:
import os, sys, subprocess
from pathlib import Path

IS_KAGGLE = os.path.exists('/kaggle/working')
IS_COLAB  = not IS_KAGGLE and os.path.exists('/content')
IS_LOCAL  = not IS_KAGGLE and not IS_COLAB

env_name = 'Kaggle' if IS_KAGGLE else 'Colab' if IS_COLAB else 'Local'
print(f"Ortam : {env_name}")

import torch
if not torch.cuda.is_available():
    if IS_LOCAL and torch.backends.mps.is_available():
        print("GPU : Apple Silicon MPS (src/train_cls.py MPS destekler ✓)")
    else:
        raise RuntimeError(
            "GPU bulunamadı!\n"
            "Kaggle: Settings → Accelerator → GPU\n"
            "Colab : Runtime → Change runtime type → GPU"
        )
else:
    print(f"GPU   : {torch.cuda.get_device_name(0)}")
    print(f"VRAM  : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
    print(f"CUDA  : {torch.version.cuda}")

---
## 1. Ortam Kurulumu (Colab için)

In [ ]:
if IS_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'kaggle'], check=True)
    try:
        from google.colab import userdata
        os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
        os.environ['KAGGLE_KEY']      = userdata.get('KAGGLE_KEY')
        print("Kaggle kimlik bilgileri Colab Secrets'tan yüklendi")
    except Exception:
        from google.colab import files
        import json as _j, shutil as _sh
        print("kaggle.json dosyasını seçin:")
        _up = files.upload()
        _kp = Path.home() / '.kaggle' / 'kaggle.json'
        _kp.parent.mkdir(parents=True, exist_ok=True)
        for fname in _up:
            _sh.move(fname, str(_kp))
        os.chmod(str(_kp), 0o600)
        _kd = _j.loads(_kp.read_text())
        os.environ['KAGGLE_USERNAME'] = _kd['username']
        os.environ['KAGGLE_KEY']      = _kd['key']

    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    print("Google Drive bağlandı")
else:
    print("Kaggle ortamı — kurulum atlandı")

---
## 2. Kütüphane Kurulumu

In [ ]:
print("Kütüphaneler kuruluyor...")
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q',
     'timm', 'pydicom', 'opencv-python-headless',
     'pandas', 'numpy', 'tqdm', 'scikit-learn'],
    check=True
)
import importlib; importlib.invalidate_caches()

import timm
print(f"timm    : {timm.__version__}")
import torch
print(f"PyTorch : {torch.__version__}")
import pydicom
print(f"pydicom : {pydicom.__version__}")

---
## 3. Konfigürasyon

In [ ]:
import os, sys, json, shutil, time, math, random
from dataclasses import dataclass
from pathlib import Path
from typing import List

import numpy as np
import pandas as pd

# ─── Kullanıcı Ayarları ────────────────────────────────────────────────────
KAGGLE_DATASET_SLUG = 'abdomen-acikveri'
KAGGLE_DATASET_ID   = 'ramazan2020/abdomen-acikveri'
GITHUB_URL          = 'https://github.com/ramazan2020/abdomen1.git'
FOLD = 0

@dataclass
class SwinConfig:
    backbone: str = 'swin_base_patch4_window12_384.ms_in22k_ft_in1k'
    num_classes: int = 6
    input_size: int = 384
    batch_size: int = 16
    epochs: int = 50
    lr: float = 2e-4
    weight_decay: float = 1e-4
    warmup_epochs: int = 3
    focal_gamma: float = 2.0
    accum_steps: int = 2
    use_focal: bool = True
    use_weighted_sampler: bool = True
    precision: str = 'bf16-mixed'
    mixup_alpha: float = 0.0

SWIN_CFG = SwinConfig()
# ──────────────────────────────────────────────────────────────────────────

SUPER_CLASSES: List[str] = [
    'acute_cholecystitis', 'kidney_ureter_stone', 'acute_pancreatitis',
    'aortic_aneurysm_dissection', 'acute_appendicitis', 'acute_diverticulitis',
]

if IS_KAGGLE:
    DATA_DIR   = Path('/kaggle/input') / KAGGLE_DATASET_SLUG
    WORK_DIR   = Path('/kaggle/working')
    TMP_DIR    = Path('/kaggle/tmp')   # 60 GB — geçici dosyalar buraya
    DRIVE_BASE = None

elif IS_COLAB:
    DATA_DIR   = Path('/content/kaggle_data')
    WORK_DIR   = Path('/content')
    DRIVE_BASE = Path('/content/drive/MyDrive/Abdomen')
    if DRIVE_BASE:
        DRIVE_BASE.mkdir(parents=True, exist_ok=True)

elif IS_LOCAL:
    try:
        from dotenv import load_dotenv; load_dotenv()
    except ImportError:
        pass
    _proje   = Path(os.environ.get('TR_ABDOMEN_PROJE',  r'D:/makale-pdf/Proje'))
    DATA_DIR = Path(os.environ.get('TR_ABDOMEN_BASE',   str(_proje / 'abdomen')))
    WORK_DIR = Path(os.environ.get('ABDOMEN_OUT_DIR',   str(_proje / 'outputs')))
    DRIVE_BASE = None

EGITIM_DIR  = Path(os.environ.get('ABDOMEN_TRAIN_DIR', str(DATA_DIR / 'Egitim Verisi')))
YARISMA_DIR = Path(os.environ.get('ABDOMEN_TEST_DIR',  str(DATA_DIR / 'Test Verisi')))
BILGI_XLSX    = Path(os.environ.get("ABDOMEN_BILGI_XLSX", str(DATA_DIR / "Bilgi.xlsx")))

if IS_LOCAL:
    SPLIT_DIR = Path(os.environ.get('ABDOMEN_SPLIT_DIR', str(WORK_DIR / 'splits')))
    NPZ_DIR   = Path(os.environ.get('ABDOMEN_CLS_DATA_DIR', str(WORK_DIR / 'cls_data')))
else:
    SPLIT_DIR = DATA_DIR / 'outputs' / 'splits'
    # NPZ (39K kesit × ~100KB) geçici → /kaggle/tmp; Colab'da working
    NPZ_DIR   = (TMP_DIR if IS_KAGGLE else WORK_DIR) / 'cls_data'

CKPT_DIR   = Path(os.environ.get('ABDOMEN_CKPT_DIR', str(WORK_DIR / 'checkpoints')))
LOG_DIR    = (TMP_DIR if IS_KAGGLE else WORK_DIR) / 'logs'  # log → tmp
OUTPUT_DIR = WORK_DIR / 'output'

for d in [NPZ_DIR, CKPT_DIR, LOG_DIR, OUTPUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# src/config.py env vars
os.environ['ABDOMEN_SPLIT_DIR']    = str(SPLIT_DIR)
os.environ['ABDOMEN_CLS_DATA_DIR'] = str(NPZ_DIR)
os.environ['ABDOMEN_CKPT_DIR']     = str(CKPT_DIR)
os.environ['ABDOMEN_LOG_DIR']      = str(LOG_DIR)
os.environ['ABDOMEN_TRAIN_DIR']    = str(EGITIM_DIR)
os.environ['ABDOMEN_TEST_DIR']     = str(YARISMA_DIR)
os.environ['ABDOMEN_OUT_DIR']      = str(WORK_DIR / 'abdomen_src_out')
os.environ['ABDOMEN_BILGI_XLSX']     = str(BILGI_XLSX)

print(f'Ortam     : {env_name}')
print(f'DATA_DIR  : {DATA_DIR}  (var={DATA_DIR.exists()})')
print(f'SPLIT_DIR : {SPLIT_DIR}  (var={SPLIT_DIR.exists()})')
print(f'NPZ_DIR   : {NPZ_DIR}  (var={NPZ_DIR.exists()})')
print(f'Backbone  : {SWIN_CFG.backbone}')
if IS_KAGGLE:
    print(f'TMP_DIR   : {TMP_DIR}  (NPZ + loglar)')

if not DATA_DIR.exists():
    raise FileNotFoundError(f'DATA_DIR bulunamadı: {DATA_DIR}')


---
## 4. Veri İndirme (Yalnızca Colab)

In [ ]:
if IS_KAGGLE:
    print("Kaggle: Dataset zaten mount edilmiş")
else:
    if DATA_DIR.exists() and EGITIM_DIR.exists():
        print(f"Veri zaten mevcut: {DATA_DIR}")
    else:
        DATA_DIR.mkdir(parents=True, exist_ok=True)
        print(f"Dataset indiriliyor: {KAGGLE_DATASET_ID}  (~30-90 dk)")
        t0 = time.time()
        r = subprocess.run(
            ['kaggle', 'datasets', 'download',
             '-d', KAGGLE_DATASET_ID,
             '-p', str(DATA_DIR), '--unzip'],
            capture_output=True, text=True
        )
        if r.returncode != 0:
            print('HATA:', r.stderr[-2000:])
            raise RuntimeError('İndirme başarısız')
        print(f"İndirildi ({(time.time()-t0)/60:.0f} dk)")

assert EGITIM_DIR.exists(), f'Egitim Verisi bulunamadı: {EGITIM_DIR}'
assert (SPLIT_DIR / 'manifest.csv').exists(), f'manifest.csv bulunamadı: {SPLIT_DIR}'
print(f'Egitim Verisi vakaları : {len(list(EGITIM_DIR.iterdir()))}')
print(f'manifest.csv satırı   : {len(pd.read_csv(SPLIT_DIR/"manifest.csv")):,}')

In [ ]:
# ── GitHub Repo / Local src ───────────────────────────────────────────────
if IS_LOCAL:
    PROJECT_ROOT = Path(__file__).resolve().parent if '__file__' in dir() else Path('.').resolve()
    REPO_DIR = PROJECT_ROOT
    print(f'Local: src/ kullanılıyor → {REPO_DIR / "src"}')
else:
    REPO_DIR = WORK_DIR / 'abdomen1'
    if not (REPO_DIR / '.git').exists():
        print(f'Klonlanıyor: {GITHUB_URL}')
        subprocess.run(
            ['git', 'clone', '--depth=1', GITHUB_URL, str(REPO_DIR)],
            check=True
        )
    else:
        print('Repo mevcut, güncelleniyor...')
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'],
                       capture_output=True)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from src.config     import ClsConfig
from src.splits     import load_fold, load_holdout
from src.datasets   import SliceMultiLabelDataset
from src.losses     import FocalBCE, compute_class_balanced_alpha
from src.train_cls  import build_model, evaluate, train_one_fold
from src.evaluation import f1_at_iou, top5_f1_mean
from src.dicom_utils import read_dicom, dicom_to_hu, hu_to_three_channel
from src.config      import DEFAULT_WINDOWS

# SwinConfig → ClsConfig (train_one_fold ClsConfig bekler)
swin_cfg = ClsConfig(
    backbone      = SWIN_CFG.backbone,
    num_classes   = SWIN_CFG.num_classes,
    input_size    = SWIN_CFG.input_size,
    batch_size    = SWIN_CFG.batch_size,
    epochs        = SWIN_CFG.epochs,
    lr            = SWIN_CFG.lr,
    weight_decay  = SWIN_CFG.weight_decay,
    warmup_epochs = SWIN_CFG.warmup_epochs,
    focal_gamma   = SWIN_CFG.focal_gamma,
    accum_steps   = SWIN_CFG.accum_steps,
    use_focal            = SWIN_CFG.use_focal,
    use_weighted_sampler = SWIN_CFG.use_weighted_sampler,
    mixup_alpha   = SWIN_CFG.mixup_alpha,
)

_train_cases = load_fold(FOLD, 'train')
_val_cases   = load_fold(FOLD, 'val')

print(f'Repo     : {REPO_DIR}')
print(f'Fold {FOLD}  : {len(_train_cases)} train / {len(_val_cases)} val vaka')
print('src modülleri ✓')

In [ ]:
# ── Veri Hazırlık Kontrolü ─────────────────────────────────────────────────
# manifest.csv + splits, Faz1_2_VeriHazirlik notebook'u tarafından üretilir.
# Mevcut değilse burada otomatik oluşturulur.

SPLIT_DIR.mkdir(parents=True, exist_ok=True)
_manifest_csv = SPLIT_DIR / 'manifest.csv'
_splits_csv   = SPLIT_DIR / 'splits.csv'

if not _manifest_csv.exists():
    print('manifest.csv bulunamadı — oluşturuluyor...')
    _bilgi = DATA_DIR / 'Bilgi.xlsx'
    if not _bilgi.exists():
        raise FileNotFoundError(
            f'Bilgi.xlsx bulunamadı: {_bilgi}\n'
            "Faz1_2_VeriHazirlik notebook'unu önce çalıştırın."
        )
    os.environ.setdefault('ABDOMEN_BILGI_XLSX', str(_bilgi))
    from src.preprocessing import build_manifest
    build_manifest(_manifest_csv)
    print(f'manifest.csv oluşturuldu ✓  ({_manifest_csv.stat().st_size/1e3:.0f} KB)')
else:
    print(f'manifest.csv mevcut ✓  ({_manifest_csv.stat().st_size/1e3:.0f} KB)')

if not _splits_csv.exists():
    print('splits.csv bulunamadı — oluşturuluyor...')
    from src.splits import make_splits
    make_splits(out_dir=SPLIT_DIR)
    print('splits.csv oluşturuldu ✓')
else:
    print('splits.csv mevcut ✓')

_fold_csv = SPLIT_DIR / f'fold{FOLD}_train.csv'
if not _fold_csv.exists():
    print(f'fold{FOLD}_train.csv eksik — make_splits yeniden çalıştırılıyor...')
    from src.splits import make_splits
    make_splits(out_dir=SPLIT_DIR)
print(f'fold{FOLD}_train.csv: {"✓ mevcut" if _fold_csv.exists() else "⚠ hâlâ eksik"}')

---
## 5. DICOM → NPZ Export

Manifest'teki her kesiti 3-kanallı float16 NPZ olarak `/kaggle/working/cls_data/` altına kaydeder.  
**Süre:** ~40–60 dakika (39K kesit)  
Session yeniden başlarsa tekrar çalıştırılması gerekir.

> `SKIP_EXPORT = True` yaparak atlayabilirsiniz (NPZ zaten mevcutsa).

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.notebook import tqdm

SKIP_EXPORT = False   # True → yeterli NPZ varsa atla

# ── Kaggle yol düzeltme ───────────────────────────────────────────────────
# manifest.csv'deki dicom_path'ler yerel makine yolunu içerir.
# Burada case+image_id'den Kaggle yolunu yeniden türetiriz.
def _export_one(row, out_dir):
    out = Path(out_dir) / f"{row['case']}_{row['image_id']}.npz"
    if out.exists():
        return True, None

    case_raw = str(row['case']).split('_', 1)[1]
    base = EGITIM_DIR if str(row['source']) == 'train' else YARISMA_DIR
    dpath = base / case_raw / f"{row['image_id']}.dcm"
    if not dpath.exists():
        return False, f'yok:{dpath}'

    try:
        ds  = read_dicom(dpath)                          # src.dicom_utils
        hu  = dicom_to_hu(ds)
        img = hu_to_three_channel(hu, DEFAULT_WINDOWS)   # HxWx3 float32
        labels = np.zeros(len(SUPER_CLASSES), dtype=np.uint8)
        sl = str(row.get('super_labels', ''))
        if sl and sl != 'nan':
            for s in sl.split(';'):
                if s.strip():
                    labels[int(s.strip())] = 1
        np.savez_compressed(
            out,
            image=img.astype(np.float16),
            labels=labels,
            case=str(row['case']),
            image_id=int(row['image_id']),
            source=str(row['source']),
        )
        return True, None
    except Exception as e:
        return False, str(e)

manifest = pd.read_csv(SPLIT_DIR / 'manifest.csv')
existing = list(NPZ_DIR.glob('*.npz'))
print(f'Manifest  : {len(manifest):,} kesit')
print(f'Mevcut NPZ: {len(existing):,}')

if SKIP_EXPORT and len(existing) >= len(manifest) * 0.95:
    print('NPZ export atlandı (yeterli dosya mevcut)')
else:
    print(f'NPZ export → {NPZ_DIR}')
    t0 = time.time()
    rows = list(manifest.itertuples(index=False))
    done = err = 0
    pbar = tqdm(total=len(rows), desc='DICOM→NPZ')
    with ThreadPoolExecutor(max_workers=8) as ex:
        futs = {ex.submit(_export_one, r._asdict(), NPZ_DIR): r for r in rows}
        for f in as_completed(futs):
            ok, e = f.result()
            if ok: done += 1
            else:  err  += 1
            pbar.update(1)
    pbar.close()
    print(f'Tamamlandı: {done:,} NPZ  ({err} hata)  {(time.time()-t0)/60:.1f} dk')

npz_count = len(list(NPZ_DIR.glob('*.npz')))
print(f'Toplam NPZ: {npz_count:,}')
if npz_count < 1000:
    raise RuntimeError(f'NPZ sayısı çok düşük ({npz_count}). DICOM dizinini kontrol edin.')

---
## 6. Yardımcı Fonksiyonlar (Dataset · Loss · Model · Metrics)

In [ ]:
# Tüm modüller cell_github'da import edildi:
#   src.train_cls  → build_model, evaluate, train_one_fold
#   src.datasets   → SliceMultiLabelDataset
#   src.losses     → FocalBCE, compute_class_balanced_alpha
#   src.splits     → load_fold, load_holdout
#   src.evaluation → f1_at_iou, top5_f1_mean

n_params = sum(p.numel() for p in build_model(swin_cfg).parameters()) / 1e6
print(f'Model    : {swin_cfg.backbone}')
print(f'Parametre: {n_params:.0f}M')
print(f'Fold {FOLD}  : {len(_train_cases)} train / {len(_val_cases)} val vaka')
print('src modülleri hazır ✓')

---
## 7. Eğitim

Session kesilirse: `SWIN_CONTINUE = True` yapıp tekrar çalıştırın.

In [ ]:
import torch
assert torch.cuda.is_available(), 'GPU gerekli!'

print(f'GPU  : {torch.cuda.get_device_name(0)}')
print(f'VRAM : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
print(f'\nEğitim başlıyor: Fold {FOLD}  |  {swin_cfg.epochs} epoch')
print(f'Checkpoint : {CKPT_DIR}/cls_fold{FOLD}/best.pth')
print()

# train_one_fold: src/train_cls.py
#   - Warmup + CosineAnnealing scheduler
#   - FocalBCE class-balanced loss
#   - AMP (bfloat16, CUDA)
#   - En iyi mAP'e göre checkpoint kaydeder → CKPT_DIR/cls_fold{FOLD}/best.pth
#   - TensorBoard log → LOG_DIR/tb/cls_fold{FOLD}/
best = train_one_fold(fold=FOLD, cfg=swin_cfg)

BEST_CKPT = CKPT_DIR / f'cls_fold{FOLD}' / 'best.pth'
print(f'\nEn iyi : epoch={best["epoch"]}  mAP={best["mAP"]:.4f}  macroF1={best["macroF1"]:.4f}')
print(f'Ckpt   : {BEST_CKPT}')

if IS_COLAB and DRIVE_BASE:
    _yd = DRIVE_BASE / 'swin_weights'
    _yd.mkdir(parents=True, exist_ok=True)
    shutil.copy2(str(BEST_CKPT), str(_yd / f'fold{FOLD}_best.pth'))
    print(f"Drive'a yedeklendi: {_yd}")

---
## 8. Validation Değerlendirmesi

In [ ]:
from torch.utils.data import DataLoader

# BEST_CKPT eğitim hücresinden gelir.
# Manuel yüklemek için:
# BEST_CKPT = CKPT_DIR / f'cls_fold{FOLD}' / 'best.pth'
assert Path(BEST_CKPT).exists(), f'Checkpoint bulunamadı: {BEST_CKPT}'

device = torch.device('cuda')
ck = torch.load(str(BEST_CKPT), map_location=device)

_eval_model = build_model(swin_cfg).to(device)
_eval_model.load_state_dict(ck['model'])
print(f"Checkpoint yüklendi → epoch={ck['epoch']}  mAP={ck['metrics']['mAP']:.4f}")

_val_ds = SliceMultiLabelDataset(_val_cases, input_size=swin_cfg.input_size)
_val_loader = DataLoader(
    _val_ds, batch_size=swin_cfg.batch_size, shuffle=False,
    num_workers=4, pin_memory=True,
)

# evaluate: src.train_cls
val_metrics = evaluate(_eval_model, _val_loader, device)

print(f'\n=== Validation Fold {FOLD} ===')
print(f'mAP    : {val_metrics["mAP"]:.4f}')
print(f'macroF1: {val_metrics["macroF1"]:.4f}')
print()
print(f'{"Sınıf":<35} {"AP":>7}  {"F1":>7}')
print('─' * 52)
for cls in SUPER_CLASSES:
    ap  = val_metrics.get(f'AP/{cls}', float('nan'))
    f1v = val_metrics.get(f'F1/{cls}', float('nan'))
    print(f'{cls:<35} {ap:>7.4f}  {f1v:>7.4f}')

---
## 9. Eğitim Grafiği

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

_log_csv = LOG_DIR / f'swin_fold{FOLD}.csv'
if not _log_csv.exists():
    print('Log CSV bulunamadı, eğitim adımını çalıştırın.')
else:
    _df = pd.read_csv(_log_csv)
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    axes[0].plot(_df['epoch'], _df['train_loss'], label='train loss')
    axes[0].set_title('Training Loss'); axes[0].set_xlabel('Epoch'); axes[0].grid(True)

    axes[1].plot(_df['epoch'], _df['mAP'],     label='mAP')
    axes[1].plot(_df['epoch'], _df['macroF1'], label='macroF1')
    axes[1].set_title('Val mAP / macroF1'); axes[1].set_xlabel('Epoch')
    axes[1].legend(); axes[1].grid(True)

    ap_cols = [c for c in _df.columns if c.startswith('AP/')]
    for col in ap_cols:
        axes[2].plot(_df['epoch'], _df[col], label=col.replace('AP/', ''))
    axes[2].set_title('Per-Class AP'); axes[2].set_xlabel('Epoch')
    axes[2].legend(fontsize=7); axes[2].grid(True)

    plt.suptitle(f'Swin Transformer — Fold {FOLD}', fontsize=12)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f'swin_fold{FOLD}_curves.png', dpi=120)
    plt.show()

---
## 10. Sonuçları Kaydet

In [ ]:
summary = {
    'fold'      : FOLD,
    'backbone'  : SWIN_CFG.backbone,
    'best_epoch': best['epoch'],
    'mAP'       : best['mAP'],
    'macroF1'   : best['macroF1'],
    'val_metrics': {
        k: round(v, 4) if isinstance(v, float) and not math.isnan(v) else v
        for k, v in val_metrics.items()
    },
    'config': SWIN_CFG.__dict__,
}
(OUTPUT_DIR / f'swin_fold{FOLD}_summary.json').write_text(
    json.dumps(summary, indent=2, ensure_ascii=False)
)

print(f'Çıktılar: {OUTPUT_DIR}')
for f in sorted(OUTPUT_DIR.glob('*')):
    print(f'  {f.name}  ({f.stat().st_size/1e3:.0f} KB)')

print()
print('=' * 50)
print(f'  Swin Transformer Fold {FOLD} — Özet')
print('=' * 50)
print(f'  Backbone   : {SWIN_CFG.backbone}')
print(f'  Best epoch : {best["epoch"]}')
print(f'  mAP        : {best["mAP"]:.4f}')
print(f'  macroF1    : {best["macroF1"]:.4f}')
print('=' * 50)

if IS_COLAB and DRIVE_BASE:
    _dst = DRIVE_BASE / 'output'
    _dst.mkdir(parents=True, exist_ok=True)
    for f in OUTPUT_DIR.glob('swin_*'):
        shutil.copy2(str(f), str(_dst / f.name))
    print(f"Drive'a kopyalandı: {_dst}")

if IS_KAGGLE:
    print('\nKaggle: Save Version → sağ üst → Save & Run All')